# 💻 Laboratório 16: Processamento de Linguagem Natural (PNL Básica)

**Disciplina:** Extração e Preparação de Dados (IBM8915)
**Professor:** Luís Aramis

**Objetivo:** Transformar dados não estruturados (textos humanos) em matrizes numéricas. Você aplicará técnicas de padronização, limpeza via Regex, remoção de *stopwords* e extração de *features* usando *Bag-of-Words* para preparar os dados de um E-commerce para Algoritmos de Machine Learning.

In [1]:
# Importação das bibliotecas essenciais
import pandas as pd
import numpy as np

# Simulando a carga de um Dataset de E-commerce (Reviews de Produtos)
# Em um cenário real, você usaria pd.read_csv('avaliacoes_ecommerce.csv')
dados_ecommerce = pd.DataFrame({
    'ID_Cliente': [1, 2, 3, 4, 5, 6],
    'Nota': [1, 5, 1, 4, 5, 2],
    'Review_Bruta': [
        "Odiei o produto, péssimo material!!! Não recomendo.",
        "Celular EXCELENTE... a bateria dura muito, amei.",
        "NÃO COMPREM! Aparelho horrível, quebrou no primeiro dia.",
        "Produto muito bom, mas a entrega atrasou um pouco.",
        "Gostei do design, bem elegante. Recomendo para todos.",
        "Decepcionado. O produto veio com defeito e o suporte é ruim."
    ]
})

print("--- Base de Dados Bruta (E-commerce) ---")
display(dados_ecommerce)

--- Base de Dados Bruta (E-commerce) ---


,ID_Cliente,Nota,Review_Bruta
0,1,1,"Odiei o produto, péssimo material!!! Não recom..."
1,2,5,"Celular EXCELENTE... a bateria dura muito, amei."
2,3,1,"NÃO COMPREM! Aparelho horrível, quebrou no pri..."
3,4,4,"Produto muito bom, mas a entrega atrasou um po..."
4,5,5,"Gostei do design, bem elegante. Recomendo para..."
5,6,2,Decepcionado. O produto veio com defeito e o s...


## Passo 1: A Faxina de Texto no Pandas (`.str`)

Algoritmos são sensíveis a variações. "Produto", "produto" e "produto!!!" são lidos como três entidades diferentes. Sua missão é usar o acessor `.str` do Pandas para unificar tudo.

**Sua Tarefa:**
1. Converta a coluna `Review_Bruta` inteira para letras minúsculas.
2. Remova toda a pontuação usando Expressões Regulares (`Regex`).

In [2]:
# 1. Transforme tudo em minúsculas (Dica: use .str.lower())
dados_ecommerce['Review_Limpa'] = dados_ecommerce['Review_Bruta'].str.lower()

# 2. Remova a pontuação (Dica: substitua [^\w\s] por espaço vazio '' usando .str.replace)
dados_ecommerce['Review_Limpa'] = dados_ecommerce['Review_Limpa'].str.replace(r'[^\w\s]', '', regex=True)

print("--- Dados Após a Faxina ---")
display(dados_ecommerce[['Nota', 'Review_Limpa']])

--- Dados Após a Faxina ---


,Nota,Review_Limpa
0,1,odiei o produto péssimo material não recomendo
1,5,celular excelente a bateria dura muito amei
2,1,não comprem aparelho horrível quebrou no prime...
3,4,produto muito bom mas a entrega atrasou um pouco
4,5,gostei do design bem elegante recomendo para t...
5,2,decepcionado o produto veio com defeito e o su...


## Passo 2: Stopwords e o Sinal Analítico

Palavras como "o", "a", "de", "para", "com" não carregam sentimento. Elas não ajudam a máquina a prever se o cliente deu Nota 1 ou Nota 5. Nós precisamos removê-las antes de criar a matriz matemática.

Nesta etapa, usaremos a biblioteca do Scikit-Learn e forneceremos uma lista customizada do idioma português para o vetorizador.

In [3]:
# Importando a ferramenta de Extração de Features de Texto do Scikit-Learn
from sklearn.feature_extraction.text import CountVectorizer

# Definindo nossa lista de Stopwords (palavras sem valor preditivo)
stopwords_pt = [
    'o', 'a', 'os', 'as', 'um', 'uns', 'uma', 'umas', 'de', 'do', 'da', 
    'dos', 'das', 'no', 'na', 'nos', 'nas', 'em', 'por', 'para', 'com', 
    'como', 'mas', 'ou', 'e', 'que', 'se', 'eu', 'ele', 'ela', 'meu', 'sua'
]

print(f"Lista de stopwords carregada com {len(stopwords_pt)} palavras conectivas para descarte.")

Lista de stopwords carregada com 32 palavras conectivas para descarte.


## Passo 3: Tradução para a Máquina (Bag-of-Words)

Esta é a fase final. O `CountVectorizer` vai ler os seus textos limpos, descartar as *stopwords*, encontrar todas as palavras únicas (*tokens*) que sobraram e transformá-las em colunas numéricas. Se o cliente usou a palavra "péssimo", ele ganha o número `1` naquela coluna. Se não usou, ganha `0`.

**Sua Tarefa:**
1. Instancie o `CountVectorizer` passando a nossa lista de `stopwords_pt`.
2. Execute o método `.fit_transform()` na coluna `Review_Limpa`.
3. Visualize a magia acontecer!

In [4]:
# 1. Instancie o vetorizador passando a lista de stopwords no parâmetro 'stop_words'
vetorizador = CountVectorizer(stop_words=stopwords_pt)

# 2. Treine e transforme a coluna de textos limpos em uma matriz esparsa
matriz_frequencia = vetorizador.fit_transform(dados_ecommerce['Review_Limpa'])

# 3. Extraindo as palavras que a máquina "aprendeu" (features)
palavras_extraidas = vetorizador.get_feature_names_out()

# Transformando a matriz matemática de volta para o Pandas para visualização humana
df_bag_of_words = pd.DataFrame(
    matriz_frequencia.toarray(), 
    columns=palavras_extraidas
)

# Concatenando a nota original ao lado da matriz matemática para ver a relação
df_final = pd.concat([dados_ecommerce[['Nota']], df_bag_of_words], axis=1)

print("--- A Matriz Matemática Final (Bag of Words) ---")
display(df_final)

--- A Matriz Matemática Final (Bag of Words) ---


,Nota,amei,aparelho,atrasou,bateria,bem,bom,celular,comprem,decepcionado,...,pouco,primeiro,produto,péssimo,quebrou,recomendo,ruim,suporte,todos,veio
0,1,0,0,0,0,0,0,0,0,0,...,0,0,1,1,0,1,0,0,0,0
1,5,1,0,0,1,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,1,0,0,0,0,0,1,0,...,0,1,0,0,1,0,0,0,0,0
3,4,0,0,1,0,0,1,0,0,0,...,1,0,1,0,0,0,0,0,0,0
4,5,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,1,0,0,1,0
5,2,0,0,0,0,0,0,0,0,1,...,0,0,1,0,0,0,1,1,0,1


## 🏆 Desafio Bônus: Analisando as Frequências

Agora que você tem um DataFrame perfeitamente numérico (`df_bag_of_words`), você consegue dizer quais são as palavras mais repetidas na sua base de clientes?

**Sua Tarefa:**
Use o método `.sum()` do Pandas no seu `df_bag_of_words` para somar a ocorrência de cada coluna (palavra). Depois, use o `.sort_values(ascending=False)` para revelar as Top 5 palavras mais faladas pelos seus clientes!

In [5]:
# Somando as ocorrências de cada palavra e ordenando as Top 5
top_palavras = df_bag_of_words.sum().sort_values(ascending=False).head(5)

print("TOP 5 Palavras Mais Usadas nos Reviews:")
print(top_palavras)

TOP 5 Palavras Mais Usadas nos Reviews:
produto      3
recomendo    2
não          2
muito        2
amei         1
dtype: int64


### Critério de Êxito para o Portfólio (GitHub):
Se você olhar para o `df_final` na célula 8, nenhuma palavra como "o", "a", ou "mas" deve existir nas colunas. Toda a matriz deve ser formada por números `0` e `1`, correlacionando palavras como "quebrou" ou "excelente" com as Notas dos clientes.

Após executar tudo e conferir se a matriz está gerada de forma limpa, não esqueça de commitar esse notebook (`lab_16_nlp.ipynb`) para fecharmos a etapa de dados não estruturados!